# Tugas 5
## Pencarian dokumen 

#### Rafly Faldiansyah Putra 210411100063

# Transformasi SVD pada TF-IDF

In [12]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics.pairwise import cosine_similarity
import re
from IPython.display import display, Markdown

# 1. Membaca Dataset
# Membaca file CSV yang berisi dataset
df = pd.read_csv("preprocessing-kompas.csv")

# Mengganti nilai NaN dengan string kosong agar tidak ada data kosong yang menyebabkan error
df['stopword_removal'] = df['stopword_removal'].fillna('')

# 2. Mengonversi Data Teks ke TF-IDF
# Menginisialisasi TfidfVectorizer dengan normalisasi L2
vectorizer = TfidfVectorizer(norm='l2')

# Menghitung nilai TF-IDF untuk setiap dokumen
tfidf_matrix = vectorizer.fit_transform(df['stopword_removal'])

# Mengubah hasil TF-IDF menjadi DataFrame untuk kemudahan akses
tfidf_df = pd.DataFrame(tfidf_matrix.toarray(), columns=vectorizer.get_feature_names_out())

# 3. Reduksi Dimensi dengan Truncated SVD
# Menginisialisasi TruncatedSVD untuk mereduksi dimensi menjadi 100 komponen (atau sesuai kebutuhan)
svd = TruncatedSVD(n_components=100, random_state=42)

# Mengaplikasikan SVD pada matriks TF-IDF untuk menghasilkan matriks dengan dimensi yang lebih rendah
svd_matrix = svd.fit_transform(tfidf_matrix)

# Menyimpan hasil SVD dalam DataFrame untuk digunakan dalam perhitungan kemiripan
svd_df = pd.DataFrame(svd_matrix)

# Menampilkan 10 baris pertama dari DataFrame hasil SVD
svd_df.head(1000)



,0,1,2,3,4,5,6,7,8,9,...,90,91,92,93,94,95,96,97,98,99
0,0.183920,0.122921,0.095698,-0.001473,0.027361,-0.021864,0.010294,-0.090545,0.086207,0.002699,...,-0.122558,0.049821,-0.121786,-0.016503,0.178932,-0.015762,0.022900,-0.025765,0.048239,-0.015331
1,0.182546,0.414930,-0.212541,0.086298,0.015497,-0.008238,0.034065,-0.057178,-0.110534,0.048016,...,-0.000934,-0.028118,-0.076313,-0.001087,-0.004250,0.013773,-0.022307,0.011682,-0.006825,-0.005323
2,0.123009,0.240731,-0.095364,0.056606,-0.039895,0.001020,-0.006324,-0.388093,-0.164308,0.060802,...,-0.017066,0.050797,0.002051,0.025975,0.054881,0.028753,-0.009979,-0.000383,-0.017222,-0.016501
3,0.131392,0.268334,-0.103155,0.025768,0.018932,-0.005625,0.011552,-0.104057,-0.038587,0.015876,...,-0.059556,-0.000691,-0.039189,0.056096,0.094552,-0.029824,0.041832,0.048761,-0.058881,-0.049125
4,0.110038,0.226463,-0.079948,0.030228,-0.021987,-0.002253,-0.002556,-0.385907,-0.137069,0.057779,...,0.054323,-0.068364,-0.010789,0.057595,-0.106894,-0.099474,0.044382,-0.000536,0.028208,-0.023351
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
195,0.345880,-0.247240,-0.223841,-0.167152,0.060953,-0.068421,-0.313469,0.065816,-0.288049,-0.176959,...,0.012766,-0.007462,0.030949,-0.008743,0.007204,0.006334,-0.020759,0.017811,0.027299,0.012747
196,0.256508,-0.139253,-0.090998,-0.036608,-0.081309,-0.057848,0.010634,0.015348,0.088427,0.097688,...,-0.106715,0.008045,0.268989,0.034824,0.046181,0.082699,0.060380,-0.149165,-0.095955,-0.087637
197,0.188451,-0.099413,-0.055812,-0.036070,-0.061292,0.019735,0.041183,-0.004119,0.056867,-0.023673,...,0.012259,-0.043574,0.016770,0.048916,0.083873,-0.003264,-0.062391,0.067613,-0.023419,0.002178
198,0.302203,-0.228062,-0.208756,-0.135833,0.177806,-0.071958,-0.257418,0.037827,-0.248653,-0.152175,...,0.034323,0.021487,0.085596,0.011443,-0.010981,0.002590,-0.050253,-0.058711,0.026890,0.080653


# Mencari Dokumen dengan Cosine Similiarity

In [15]:
# Fungsi untuk mencari dokumen terkait
def search_documents(query, top_n=100, similarity_threshold=0.5):
    # 4. Menghitung TF-IDF untuk kalimat inputan
    query_tfidf = vectorizer.transform([query])
    
    # 5. Mengurangi dimensi kalimat inputan
    query_svd = svd.transform(query_tfidf)
    
    # 6. Menghitung kemiripan kosinus
    cosine_similarities = cosine_similarity(query_svd, svd_df)
    
    # 7. Menyaring hasil kemiripan berdasarkan threshold
    similar_indices = [
        idx for idx, score in enumerate(cosine_similarities[0])
        if score >= similarity_threshold
    ]
    
    # 8. Mengambil dokumen terkait berdasarkan threshold
    related_docs = df.iloc[similar_indices]
    
    # Membatasi jumlah dokumen yang diambil
    return related_docs.head(top_n)

# Contoh penggunaan - Input dari pengguna
input_query = input("Masukkan kalimat pencarian Anda: ")
related_documents = search_documents(input_query)

# Menampilkan dokumen terkait dengan detail yang diminta
print("=" * 50)
print(f"{'Dokumen Terkait':^50}")
print("=" * 50)
displayed_count = 0

for index, row in related_documents.iterrows():
    print(f"Dokumen #{displayed_count + 1}")
    print(f"Judul    : {row['judul']}")
    print(f"Tanggal  : {row['tanggal']}")
    print(f"Kategori : {row['kategori']}")
    print(f"Isi Berita:\n{row['isi_berita']}")
    print("-" * 50)  # Pemisah antar dokumen
    displayed_count += 1

# Menampilkan jumlah dokumen yang ditampilkan
print(f"\nJumlah dokumen yang ditampilkan: {displayed_count}")
print("=" * 50)


Masukkan kalimat pencarian Anda:  dito


                 Dokumen Terkait                  
Dokumen #1
Judul    : Menpora Dito Ariotedjo Ungkap Pesan Prabowo soal Olahraga: Tak Hanya Kompetisi, tetapi Diplomasi
Tanggal  : 19/10/2024, 21:08 WIB
Kategori : Olahraga
Isi Berita:
 JAKARTA, KOMPAS.com - Menteri Pemuda dan Olahraga (Menpora) Dito Ariotedjo mengungkapkan pesan atau keinginan presiden terpilih Prabowo Subianto untuk dunia olahraga Indonesia ke depan.  Ia menyebutkan, dalam diskusinya dengan Prabowo, olahraga tak boleh hanya dianggap sebagai kompetisi belaka. Tetapi juga bagian dari diplomasi dengan negara lain.  “Pak Prabowo menekankan bahwa olahraga bukan sekadar kompetisi, tapi juga sebagai sarana memperkuat citra dan prestise negara di mata dunia,” ujar Dito dalam keterangannya, Sabtu (19/10/2024). Baca juga: Utusan Khusus Negara Sahabat Tiba di Jakarta untuk Hadiri Pelantikan Prabowo-Gibran  Ia menyebutkan, saat ini Indonesia tengah berupaya merajut kerja sama olahraga dengan beberapa negara.  Hal itu, menjadi sal